#                              **AI4005: Mini Project**
##                      **ANN parameterized Fuzzy C-Means Clustering Algorithm**
### Anirudh Srinivasan (CS20BTECH11059) | Ajit Shankar (ES20BTECH11003)

In [ ]:
!pip install deap

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.4/135.4 kB 2.9 MB/s eta 0:00:00


In [ ]:
!pip install tqdm

In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
import numpy as np
import tensorflow as tf
from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from scipy.spatial.distance import cdist
from deap import base, creator, tools, algorithms
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs

## Initial Attempts

In [ ]:
# Load and prepare the Iris dataset
iris = datasets.load_iris()
X = iris.data
y = iris.target

# Normalize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Define the ANN and FCM evaluation function
def evaluate(individual, X_train, X_test):
    num_hidden_layers = individual[0]
    num_nodes_per_layer = individual[1]
    activation_index = individual[2]  # 0 for relu, 1 for sigmoid
    num_clusters = individual[3]

    activations = ['relu', 'sigmoid']
    activation_function = activations[activation_index]

    # Build the ANN model
    model = tf.keras.Sequential()
    model.add(tf.keras.layers.InputLayer(input_shape=(X_train.shape[1],)))
    for _ in range(num_hidden_layers):
        model.add(tf.keras.layers.Dense(num_nodes_per_layer, activation=activation_function))
    model.add(tf.keras.layers.Dense(num_clusters, activation='softmax'))  # Output layer for memberships
    model.compile(optimizer='adam', loss='mse')

    # Train the model
    model.fit(X_train, X_train, epochs=30, batch_size=32, verbose=0)

    # Predict memberships on the test set
    mf = model.predict(X_test)

    # Apply FCM clustering on the output of the ANN
    centroids = np.dot(mf.T, X_test) / np.sum(mf, axis=0)[:, None]
    distances = cdist(X_test, centroids, 'euclidean')
    J = np.sum((mf ** 2) * (distances ** 2))  # FCM objective

    return J,

# Genetic Algorithm Setup using DEAP
creator.create("FitnessMin", base.Fitness, weights=(-1.0,))
creator.create("Individual", list, fitness=creator.FitnessMin)

toolbox = base.Toolbox()
toolbox.register("attr_num_layers", np.random.randint, 1, 4)
toolbox.register("attr_num_nodes", np.random.randint, 5, 20)
toolbox.register("attr_activation", np.random.randint, 0, 2)
toolbox.register("attr_clusters", np.random.randint, 2, 5)  # 2 to 4 clusters

toolbox.register("individual", tools.initCycle, creator.Individual,
                 (toolbox.attr_num_layers, toolbox.attr_num_nodes, toolbox.attr_activation, toolbox.attr_clusters), n=1)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)

X_train, X_test = train_test_split(X_scaled, test_size=0.2, random_state=42)

def eval_wrapper(X_train, X_test):
    return lambda ind: evaluate(ind, X_train, X_test)

toolbox.register("evaluate", eval_wrapper(X_train, X_test))
toolbox.register("mate", tools.cxTwoPoint)
toolbox.register("mutate", tools.mutUniformInt, low=[1, 5, 0, 2], up=[3, 20, 1, 4], indpb=0.1)
toolbox.register("select", tools.selTournament, tournsize=3)

# Run the Genetic Algorithm
population = toolbox.population(n=50)
hof = tools.HallOfFame(1)
stats = tools.Statistics(lambda ind: ind.fitness.values)
stats.register("avg", np.mean)
stats.register("min", np.min)
stats.register("max", np.max)

result = algorithms.eaSimple(population, toolbox, cxpb=0.5, mutpb=0.2, ngen=50, stats=stats, halloffame=hof)

# Print the best model found
best_individual = hof.items[0]
print('Best Model:', best_individual)
print('Best Fitness:', best_individual.fitness.values[0])

### Using Genetic Algorithm to Optimize Neural Network Architecture with Fuzzy C-Means Clustering

This Python script demonstrates how to use a genetic algorithm (GA) implemented with DEAP (Distributed Evolutionary Algorithms in Python) to optimize the architecture of a neural network (ANN) for clustering using Fuzzy C-Means (FCM) clustering. Let's break down the script and its components:

#### Importing Libraries
- `numpy`, `tensorflow`, and `sklearn`: Standard libraries for numerical computing, machine learning, and data preprocessing.
- `datasets` from `sklearn`: For loading the Iris dataset.
- `StandardScaler` from `sklearn.preprocessing`: For normalizing input features.
- `base`, `creator`, `tools`, `algorithms` from `deap`: DEAP library components for genetic algorithm implementation.
- `cdist` from `scipy.spatial.distance`: For computing distances between data points.

#### Loading and Preparing the Iris Dataset
- The Iris dataset is loaded using `datasets.load_iris()`.
- Features and target labels are extracted from the dataset.
- Features are normalized using `StandardScaler`.

#### Evaluation Function
- `evaluate()` function takes an individual (candidate solution) representing the neural network architecture and evaluates its performance.
- It defines an ANN with the specified architecture and activation functions.
- The ANN is trained on the training data (`X_train`) and then used to predict memberships on the test set (`X_test`).
- FCM clustering is applied to the output of the ANN to compute the objective function value.
- The objective function value (FCM objective) is returned as a tuple.

#### Genetic Algorithm Setup
- Fitness and individual classes are defined using `creator`.
- `toolbox` is created to register genetic operators like initialization, evaluation, mating, mutation, and selection.
- Initialization methods are registered for each parameter of the individual.
- A wrapper function `eval_wrapper()` is defined to facilitate the evaluation of individuals by passing the training and test data.
- Genetic operators (`mate`, `mutate`, `select`) and the evaluation function are registered with the toolbox.

#### Running the Genetic Algorithm
- A population of individuals is initialized using `toolbox.population()`.
- Hall of Fame (`hof`) and statistics are defined to store the best individuals and track statistics during the evolutionary process.
- The `algorithms.eaSimple()` function is used to run the genetic algorithm with specified parameters (`cxpb`, `mutpb`, `ngen`).
- The best individual found is extracted from the Hall of Fame.
- Finally, the best model and its fitness value are printed.

#### Explanation
This script combines genetic algorithms with neural networks and FCM clustering to optimize the architecture of an ANN for clustering tasks. It defines a set of parameters for the ANN architecture (number of layers, nodes per layer, activation function, number of clusters) as individuals in a genetic algorithm. The genetic algorithm evolves these individuals over generations, aiming to minimize the FCM objective function, which measures the quality of clustering.

The script initializes a population of candidate solutions (individuals), evaluates them using the defined evaluation function, applies genetic operators (crossover and mutation) to generate new individuals, and selects the fittest individuals to proceed to the next generation. This process continues for a specified number of generations.

At the end of the evolutionary process, the best individual (representing the best neural network architecture) is extracted from the Hall of Fame, and its architecture and fitness value are printed.

In [ ]:
# Set random seed for reproducibility
np.random.seed(42)

# Generate synthetic clustering dataset
X, _ = make_blobs(n_samples=300, centers=4, cluster_std=0.60, random_state=0)

# Standardize the dataset
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split the dataset into training and testing sets
X_train, X_test = train_test_split(X_scaled, test_size=0.2, random_state=42)

In [ ]:
# Set random seed for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# Define the evaluation function for GA
def evaluate(individual, X_train, X_test):
    num_hidden_layers = individual[0]
    num_nodes_per_layer = individual[1]
    activation_index = individual[2]  # 0 for relu, 1 for sigmoid
    num_clusters = individual[3]

    activations = ['relu', 'sigmoid']
    activation_function = activations[activation_index]

    # Initialize KMeans for training target preparation
    kmeans = KMeans(n_clusters=num_clusters, random_state=42).fit(X_train)
    y_train_one_hot = tf.keras.utils.to_categorical(kmeans.labels_, num_clusters)

    # Build the ANN model
    model = tf.keras.Sequential()
    model.add(tf.keras.layers.InputLayer(input_shape=(X_train.shape[1],)))
    for _ in range(num_hidden_layers):
        model.add(tf.keras.layers.Dense(num_nodes_per_layer, activation=activation_function))
    model.add(tf.keras.layers.Dense(num_clusters, activation='softmax'))  # Output layer for memberships
    model.compile(optimizer='adam', loss='categorical_crossentropy')

    # Train the model using one-hot encoded labels
    model.fit(X_train, y_train_one_hot, epochs=1, batch_size=32, verbose=0)

    # Predict memberships on the test set
    mf = model.predict(X_test)

    # Apply FCM clustering on the output of the ANN
    centroids = np.dot(mf.T, X_test) / np.sum(mf, axis=0)[:, None]
    distances = cdist(X_test, centroids, 'euclidean')
    J = np.sum((mf ** 2) * (distances ** 2))  # FCM objective function

    return J,

# Setup DEAP for Genetic Algorithm
creator.create("FitnessMin", base.Fitness, weights=(-1.0,))
creator.create("Individual", list, fitness=creator.FitnessMin)

toolbox = base.Toolbox()
toolbox.register("attr_num_layers", np.random.randint, 1, 4)
toolbox.register("attr_num_nodes", np.random.randint, 5, 20)
toolbox.register("attr_activation", np.random.randint, 0, 2)
toolbox.register("attr_clusters", np.random.randint, 2, 5)  # Allowing for 2 to 4 clusters

toolbox.register("individual", tools.initCycle, creator.Individual,
                 (toolbox.attr_num_layers, toolbox.attr_num_nodes, toolbox.attr_activation, toolbox.attr_clusters), n=1)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)

toolbox.register("evaluate", evaluate, X_train=X_train, X_test=X_test)
toolbox.register("mate", tools.cxTwoPoint)
toolbox.register("mutate", tools.mutUniformInt, low=[1, 5, 0, 2], up=[3, 20, 1, 4], indpb=0.1)
toolbox.register("select", tools.selTournament, tournsize=3)

# Run the Genetic Algorithm
population = toolbox.population(n=50)
hof = tools.HallOfFame(1)
stats = tools.Statistics(lambda ind: ind.fitness.values)
stats.register("avg", np.mean)
stats.register("min", np.min)
stats.register("max", np.max)

result = algorithms.eaSimple(population, toolbox, cxpb=0.5, mutpb=0.2, ngen=100, stats=stats, halloffame=hof)

# Display the best model found
best_individual = hof.items[0]
print('Best Model:', best_individual)
print('Best Fitness:', best_individual.fitness.values[0])

2/2 [==============================] - 0s 5ms/step


2/2 [==============================] - 0s 7ms/step


2/2 [==============================] - 0s 5ms/step
gen	nevals	avg    	min    	max    
0  	50    	45.1622	30.1609	72.5574
2/2 [==============================] - 0s 6ms/step
1  	34    	36.1146	30.0992	53.6842
2/2 [==============================] - 0s 6ms/step
2  	28    	32.169 	30.0506	60.2624
2/2 [==============================] - 0s 4ms/step
3  	26    	31.878 	30.0128	60.262 
2/2 [==============================] - 0s 5ms/step
4  	35    	31.4186	29.8952	50.3002
2/2 [==============================] - 0s 5ms/step
5  	30    	30.7211	29.8952	34.2301
2/2 [==============================] - 0s 5ms/step
6  	28    	31.514 	29.9665	60.5348
2/2 [==============================] - 0s 5ms/step
7  	29    	31.29  	29.8713	60.951 
2/2 [==============================] - 0s 5ms/step
8  	26    	30.8115	29.8713	36.8063
2/2 [==============================] - 0s 10ms/step
9  	33    	31.0624	29.8713	44.2764
2/2 [==============================] - 0s 4ms/step
10 	25    	31.0636	29.9809	49.0087
2/2 [=============